In [26]:
from pathlib import Path
import pandas as pd
import numpy as np
path = Path("data")

In [27]:
from sklearn.model_selection import train_test_split
train_df = pd.read_csv(path / "train.csv", index_col="Id")
test_df = pd.read_csv(path / "test.csv", index_col="Id")

GYB_max = train_df["GarageYrBlt"].max()
YS_max  = train_df["YrSold"].max()

train_df["GarageYrBlt"] = GYB_max - train_df["GarageYrBlt"]
test_df["GarageYrBlt"]  = GYB_max - test_df["GarageYrBlt"]

train_df["YrSold"] = YS_max - train_df["YrSold"]
test_df["YrSold"]  = YS_max - test_df["YrSold"]


X = train_df.drop(columns=["SalePrice", "GarageFinish", "Street",
                           "Alley", "Utilities", "Condition2", "PoolQC",
                           "LotFrontage","MasVnrArea", "MasVnrType",
                           "Heating", "LowQualFinSF", "MiscFeature",
                           "RoofMatl"])
test_df = test_df.drop(columns=["GarageFinish", "Street",
                           "Alley", "Utilities", "Condition2", "PoolQC",
                           "LotFrontage","MasVnrArea", "MasVnrType",
                           "Heating", "LowQualFinSF", "MiscFeature",
                           "RoofMatl"])

y = train_df["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.15)

del X, y

In [28]:
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, MinMaxScaler
from category_encoders import CountEncoder

def apply_custom_mappings(train_df, test_df, test):
    mappings = {
        "GarageQual": lambda x: 0 if x == "TA" else 1,
        "GarageCond": lambda x: 0 if x == "TA" else 1,
        "PavedDrive": lambda x: 0 if x == "Y" else 1,
        "EnclosedPorch": lambda x: 0 if x == 0 else 1,
        "3SsnPorch": lambda x: 0 if x == 0 else 1,
        "ScreenPorch": lambda x: 0 if x == 0 else 1,
        "PoolArea": lambda x: 0 if x == 0 else 1,
        "MiscVal": lambda x: 0 if x == 0 else 1,
        "SaleType": lambda x: 0 if x == "WD" else 1,
        "LotShape": lambda x: 0 if x == "Reg" else (1 if x == "IR1" else 2),
        "LandContour": lambda x: 0 if x == "Lvl" else 1,
        "LandSlope": lambda x: 0 if x == "Gtl" else 1,
        "Condition1": lambda x: 0 if x == "Norm" else 1,
        "HouseStyle": lambda x: 0 if x == "1Story" else (1 if x == "2Story" else 2),
        "MSZoning": lambda x: 0 if x == "RL" else (1 if x == "RM" else 2),
        "Fence": lambda x: 0 if pd.isna(x) else 1,
        "FireplaceQu": lambda x: 0 if pd.isna(x) else 1,
        "BsmtExposure": lambda x: 2 if pd.isna(x) else (0 if x == "No" else 1),
        "BsmtCond": lambda x: 0 if x == "TA" else 1,
        "BsmtQual": lambda x: 0 if x == "TA" else (1 if x == "Gd" else 2),
        "BsmtFinType2": lambda x: 0 if x == "Unf" else 1,
        "BsmtFinSF2": lambda x: 0 if x == 0 else 1,
        "HeatingQC": lambda x: 0 if x == "Ex" else (1 if x == "TA" else (2 if x == "Gd" else 3)),
        "CentralAir": lambda x: 1 if x == "Y" else 0,
        "Electrical": lambda x: 0 if x == "SBrkr" else 1,
        "KitchenQual": lambda x: 0 if x == "TA" else (1 if x == "Gd" else 2),
        "Functional": lambda x: 0 if x == "Typ" else 1,
        "RoofStyle": lambda x: 0 if x == "Gable" else (1 if x == "Hip" else 2),
        "ExterCond": lambda x: 0 if x == "TA" else (1 if x == "Gd" else 2),
        "ExterQual": lambda x: 0 if x == "TA" else (1 if x == "Gd" else 2),
    }
    
    for col, func in mappings.items():
        train_df[col] = train_df[col].apply(func)
        test_df[col] = test_df[col].apply(func)
        test[col] = test[col].apply(func)
    return train_df, test_df, test

X_train, X_test, test_df = apply_custom_mappings(X_train, X_test, test_df)

countercols = ["Neighborhood", "BsmtFinType1", "Exterior1st", "Exterior2nd", "Foundation"]
def encoder(tr, va, te):
    name = tr.name
    ec = CountEncoder(cols=[name])
    tr_s = ec.fit_transform(tr.to_frame())[name].astype(float)
    m = float(tr_s.max()) if tr_s.max() != 0 else 1.0
    va_s = ec.transform(va.to_frame())[name].astype(float).fillna(0.0)
    te_s = ec.transform(te.to_frame())[name].astype(float).fillna(0.0)
    return tr_s/m, va_s/m, te_s/m

oscols = ["GarageType", "SaleCondition", "LotConfig", "BldgType"]
def oscale(tr, va, te):
    tr_n, va_n, te_n = tr.to_numpy().reshape(-1,1), va.to_numpy().reshape(-1,1), te.to_numpy().reshape(-1,1)
    oscaler = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    tr_n = oscaler.fit_transform(tr_n).ravel()
    va_n = oscaler.transform(va_n).ravel()
    te_n = oscaler.transform(te_n).ravel()
    return pd.Series(tr_n, index=tr.index), pd.Series(va_n, index=va.index), pd.Series(te_n, index=te.index)

mcols = ["MoSold", "YrSold", "OverallQual", "OverallCond", "KitchenQual"]
def mmax(tr, va, te):
    if tr.isna().any().any() or va.isna().any().any() or te.isna().any().any():
        raise ValueError(f"NaN value found in dataset tr:{tr.isna().any().any()} va{va.isna().any().any()} te: {te.isna().any().any()}")
    tr /= tr.max()
    va /= tr.max()
    te /= tr.max()

    return tr, va, te
scols = ["WoodDeckSF", "OpenPorchSF", "LotArea", 
         "MSSubClass", "BsmtUnfSF", "TotalBsmtSF",
         "1stFlrSF", "GrLivArea", "GarageArea"]
def sscaler(tr, va, te):
    ss = StandardScaler()
    tr = ss.fit_transform(tr)
    va = ss.transform(va)
    te = ss.transform(te)
    return tr.ravel(), va.ravel(), te.ravel()

X_train['LotArea'] = np.log1p(X_train['LotArea'])
X_test['LotArea'] = np.log1p(X_test['LotArea'])
test_df['LotArea'] = np.log1p(test_df['LotArea'])

X_train['2ndFlrSF'] = np.log1p(X_train['2ndFlrSF'])
X_test['2ndFlrSF'] = np.log1p(X_test['2ndFlrSF'])
test_df['2ndFlrSF'] = np.log1p(test_df['2ndFlrSF'])

X_train['BsmtFinType1'] = X_train['BsmtFinType1'].fillna('NoBsmt')
X_test['BsmtFinType1'] = X_test['BsmtFinType1'].fillna('NoBsmt')
test_df['BsmtFinType1'] = test_df['BsmtFinType1'].fillna('NoBsmt')

X_train['GarageYrBlt'] = X_train['GarageYrBlt'].fillna(-1)
X_test['GarageYrBlt'] = X_test['GarageYrBlt'].fillna(-1)
test_df['GarageYrBlt'] = test_df['GarageYrBlt'].fillna(-1)

X_train['GarageYrBlt'] = (X_train['GarageYrBlt'] + 1e-7) / X_train['GarageYrBlt'].max()
X_test['GarageYrBlt']  = (X_test['GarageYrBlt']  + 1e-7) / X_train['GarageYrBlt'].max()
test_df['GarageYrBlt']  = (test_df['GarageYrBlt']  + 1e-7) / X_train['GarageYrBlt'].max()

X_train['GarageType'] = X_train['GarageType'].fillna("Nogrg")
X_test['GarageType'] = X_test['GarageType'].fillna("Nogrg")
test_df['GarageType'] = test_df['GarageType'].fillna("Nogrg")

for col in oscols:
    X_train[col], X_test[col], test_df[col] = oscale(X_train[col], X_test[col], test_df[col])
for col in countercols:
    X_train[col], X_test[col], test_df[col] = encoder(X_train[col], X_test[col], test_df[col])
for col in scols:
    X_train[col], X_test[col], test_df[col] = sscaler(X_train[col].to_numpy().reshape(-1, 1), 
                                        X_test[col].to_numpy().reshape(-1, 1),
                                        test_df[col].to_numpy().reshape(-1, 1))
for col in mcols:
    X_train[col], X_test[col], test_df[col] = mmax(X_train[col], X_test[col], test_df[col])

In [29]:
test_df = test_df[X_train.columns]

In [30]:
X_train.isna().any().any()

np.False_

In [31]:
test_df.columns[test_df.isna().any()]

Index(['BsmtFinSF1', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath',
       'BsmtHalfBath', 'GarageCars', 'GarageArea'],
      dtype='object')

In [32]:
bad_rows = test_df.index[test_df.isna().any(axis=1)]
bad_cols = test_df.columns[test_df.isna().any()]
fill_vals = X_train.median(numeric_only=True)
test_df[bad_cols] = test_df[bad_cols].fillna(fill_vals[bad_cols])

In [33]:
from sklearn.preprocessing import StandardScaler

def zscore_if_median_gt1(X_train, X_valid, X_test):
    scaler = StandardScaler()
    numeric_cols = X_train.select_dtypes(include=[np.number]).columns
    
    cols_to_scale = [col for col in numeric_cols if X_train[col].median() > 1]
    
    X_train_scaled = X_train.copy()
    X_valid_scaled = X_valid.copy()
    X_test_scaled = X_test.copy()
    
    X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
    X_valid_scaled[cols_to_scale] = scaler.transform(X_valid[cols_to_scale])
    X_test_scaled[cols_to_scale]  = scaler.transform(X_test[cols_to_scale])
    
    return X_train_scaled, X_valid_scaled, X_test_scaled, cols_to_scale

# Kullanım
X_train, X_test, test_df, scaled_cols = zscore_if_median_gt1(X_train, X_test, test_df)
print("Z-score uygulanan kolonlar:", scaled_cols)


Z-score uygulanan kolonlar: ['LotConfig', 'YearBuilt', 'YearRemodAdd', 'BsmtFinSF1', 'FullBath', 'BedroomAbvGr', 'TotRmsAbvGrd', 'GarageCars', 'SaleCondition']


In [34]:
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error, mean_squared_error

params = dict(
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=2.0,
    objective="reg:squarederror",
    tree_method="hist",
    device = "cuda",
    random_state=42,
    eval_metric="rmse",
)
y_train = np.log1p(y_train)
y_test = np.log1p(y_test)

dtrain = xgb.DMatrix(X_train, label=y_train)
dvalid = xgb.DMatrix(X_test, label=y_test)
dtest = xgb.DMatrix(test_df)

model = xgb.train(
    params,
    dtrain,
    num_boost_round=20000,
    evals=[(dtrain, "train"), (dvalid, "valid")],
    early_stopping_rounds=300,
    verbose_eval=100
)

pred_log = model.predict(dvalid, iteration_range=(0, model.best_iteration+1))
rmsle = root_mean_squared_error(y_test, pred_log)
print("RMSLE:", rmsle)

[0]	train-rmse:0.39024	valid-rmse:0.39617
[100]	train-rmse:0.10764	valid-rmse:0.29219
[200]	train-rmse:0.07979	valid-rmse:0.28732
[300]	train-rmse:0.06828	valid-rmse:0.28723
[400]	train-rmse:0.05990	valid-rmse:0.28503
[452]	train-rmse:0.05591	valid-rmse:0.28389
RMSLE: 0.28387194333908744


In [35]:
preds = np.expm1(model.predict(dtest, iteration_range=(0, model.best_iteration+1)))
sub = pd.DataFrame({"Id": test_df.index, "SalePrice": preds})
sub.to_csv("submission.csv", index=False)
print(pd.read_csv("submission.csv").head()) 

     Id  SalePrice
0  1461  170915.95
1  1462  196586.67
2  1463  212406.53
3  1464  209697.80
4  1465  201637.06


In [36]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

Xtr = X_train.to_numpy(dtype=np.float32)
Xva = X_test.to_numpy(dtype=np.float32)
Xte = test_df.to_numpy(dtype=np.float32)

ytr = (y_train.to_numpy()).astype(np.float32).reshape(-1, 1)
yva = (y_test.to_numpy()).astype(np.float32).reshape(-1, 1)

dl_tr = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr)),
                   batch_size=256, shuffle=True, drop_last=False)
dl_va = DataLoader(TensorDataset(torch.from_numpy(Xva), torch.from_numpy(yva)),
                   batch_size=256, shuffle=False)

Xte_t = torch.from_numpy(Xte)

In [37]:
class ResBlock(nn.Module):
    def __init__(self, dim, p=0.2):
        super().__init__()
        self.res = nn.Sequential(
            nn.Linear(dim, dim),
            nn.SiLU(),
            nn.BatchNorm1d(dim),
            nn.Dropout(p)
        )
    def forward(self, x):
        return x + self.res(x)

In [ ]:
class FFNN(nn.Module):
    def __init__(self, in_dim, widths=(1024, 512, 256), p=0.2):
        super().__init__()
        layers = []
        last = in_dim
        
        for w in widths:
            layers += [
                nn.Linear(last, w),
                nn.SiLU(),
                nn.BatchNorm1d(w), # TODO: Layernorm
                """
                Girişten çıkışa “wide” (lineer) yol yok → MLP’nin basit lineer ilişkileri yakalaması zorlaşıyor.

LR/regularization agresif olabilir → loss patinaj yapar; outlier’lar için Huber (SmoothL1) daha iyi.
                BN’leri LayerNorm yap.

Mimarîyi 512→256 gibi küçült, input→output wide skip ekle.

Huber loss (log-space), LR=5e-4, weight_decay=1e-4, grad clip=1.0.

Girişte StandardScaler (train’de fit); NaN/inf kesin temizle.

Early stopping + seed sabitle.


                """
                nn.Dropout(p),
                ResBlock(w, p)
            ]
            last = w
            
        self.backbone = nn.Sequential(*layers)
        self.head = nn.Sequential(
            nn.Linear(last, 128),
            nn.SiLU(),
            nn.Dropout(p/2),
            nn.Linear(128, 1)
        )
        
    def forward(self, x):
        h = self.backbone(x)
        return self.head(h)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FFNN(in_dim=X_train.shape[1], widths=(1024,512,256), p=0.2).to(device)

opt   = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3)
lossf = nn.MSELoss()

In [39]:
best, patience, pat = 1e9, 10, 0
for epoch in range(80):
    model.train()
    for xb, yb in dl_tr:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        pred = model(xb)
        loss = lossf(pred, yb)
        loss.backward()
        opt.step()

    # valid rmsle
    model.eval()
    se, n = 0.0, 0
    with torch.no_grad():
        for xb, yb in dl_va:
            xb, yb = xb.to(device), yb.to(device)
            pv = model(xb)
            se += torch.sum((pv - yb)**2).item()
            n  += yb.shape[0]
    rmsle = (se / n) ** 0.5
    sched.step(rmsle)
    print(f"epoch {epoch:02d} rmsle={rmsle:.5f}")

    if rmsle + 1e-6 < best:
        best, pat = rmsle, 0
        torch.save(model.state_dict(), "best_ffnn.pt")
    else:
        pat += 1
        if pat >= patience:
            print("early stop."); break 


epoch 00 rmsle=11.67989
epoch 01 rmsle=9.53162
epoch 02 rmsle=5.43621
epoch 03 rmsle=4.46261
epoch 04 rmsle=5.35598
epoch 05 rmsle=4.28294
epoch 06 rmsle=3.75567
epoch 07 rmsle=3.86330
epoch 08 rmsle=3.86267
epoch 09 rmsle=3.67239
epoch 10 rmsle=3.54009
epoch 11 rmsle=3.40926
epoch 12 rmsle=3.55410
epoch 13 rmsle=3.45623
epoch 14 rmsle=3.38762
epoch 15 rmsle=3.46691
epoch 16 rmsle=3.28446
epoch 17 rmsle=3.23814
epoch 18 rmsle=3.36318
epoch 19 rmsle=3.24220
epoch 20 rmsle=3.08350
epoch 21 rmsle=3.08889
epoch 22 rmsle=3.10442
epoch 23 rmsle=2.89078
epoch 24 rmsle=2.94320
epoch 25 rmsle=3.08773
epoch 26 rmsle=2.96140
epoch 27 rmsle=2.98053
epoch 28 rmsle=2.94484
epoch 29 rmsle=2.88244
epoch 30 rmsle=2.82774
epoch 31 rmsle=2.76949
epoch 32 rmsle=2.69335
epoch 33 rmsle=2.61727
epoch 34 rmsle=2.61132
epoch 35 rmsle=2.51984
epoch 36 rmsle=2.46195
epoch 37 rmsle=2.48540
epoch 38 rmsle=2.47231
epoch 39 rmsle=2.43052
epoch 40 rmsle=2.46316
epoch 41 rmsle=2.45830
epoch 42 rmsle=2.42133
epoch 43 r

In [ ]:
model.load_state_dict(torch.load("best_ffnn.pt"))
model.eval()
with torch.no_grad():
    preds_log = model(Xte_t.to(device).float()).cpu().numpy().ravel()
preds = np.expm1(preds_log)

sub = pd.DataFrame({"Id": test_df.index, "SalePrice": preds})
sub.to_csv("submission_ffnn.csv", index=False)
print(sub.head())


     Id      SalePrice
0  1461   59887.191406
1  1462   48809.746094
2  1463  105371.007812
3  1464  102549.898438
4  1465   66263.398438
